In [2]:
import datetime
from IPython.display import display
import pysacsnow
import shutil
from pathlib import Path
from pysacsnow.base.data_models import ConfigFile

In [3]:
base_dir = Path('/home/kvanwerkhoven/projects/pysac')
region = 'hawaii'

study_dir = Path(base_dir, region)
# data_dir = Path(study_dir, 'data')
# read_dir = Path(study_dir, 'pyMARRMot_input')
# forcing_dir = Path(data_dir, 'forcing')
# obs_dir = Path(data_dir, 'observations')

In [4]:
# Setting up test study directory for new model run
#shutil.rmtree(study_dir, ignore_errors=True)
study_dir.mkdir(parents=True, exist_ok=True)
print(f"Test study directory: {study_dir}")

Test study directory: /home/kvanwerkhoven/projects/pysac/hawaii


### Create directory structure for model files

NOTE: This is not mandatory, you can use any directory structure you like since the model config files 
take the full path to the model files. However, this is a convenient way to set up the directory structure
for the model files and there is helper function.

In [5]:
# Using the model_utils module to create directory structure for model files
pysacsnow.model_utils.create_new_models(study_dir)


Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/data/forcing
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/data/output
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/sac
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/snow17
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/unithg
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/multi-model


In [23]:
# Check if folders were created
for p in study_dir.rglob('*'):
    if p.is_dir():
        print(p)

/home/kvanwerkhoven/projects/pysac/hawaii/pyMARRMot_input
/home/kvanwerkhoven/projects/pysac/hawaii/configurations
/home/kvanwerkhoven/projects/pysac/hawaii/data
/home/kvanwerkhoven/projects/pysac/hawaii/.ipynb_checkpoints
/home/kvanwerkhoven/projects/pysac/hawaii/pyMARRMot_input/pysac_input
/home/kvanwerkhoven/projects/pysac/hawaii/pyMARRMot_input/.ipynb_checkpoints
/home/kvanwerkhoven/projects/pysac/hawaii/configurations/snow17
/home/kvanwerkhoven/projects/pysac/hawaii/configurations/unithg
/home/kvanwerkhoven/projects/pysac/hawaii/configurations/sac
/home/kvanwerkhoven/projects/pysac/hawaii/configurations/multi-model
/home/kvanwerkhoven/projects/pysac/hawaii/data/observations
/home/kvanwerkhoven/projects/pysac/hawaii/data/output
/home/kvanwerkhoven/projects/pysac/hawaii/data/forcing
/home/kvanwerkhoven/projects/pysac/hawaii/pyMARRMot_input/pysac_input/.ipynb_checkpoints
/home/kvanwerkhoven/projects/pysac/hawaii/configurations/sac/.ipynb_checkpoints
/home/kvanwerkhoven/projects/pysac

As you can see, the model_utils module created the necessary directories for the model files
however, the model files themselves are not created. You can create them manually from other examples files, 
or use the model_utils module to help create the model configuration yaml files.

The parameter input files are simple json inputs so you can use code or anything to make those, and the forcing data can similarly be created in any method as long the format meets the requirements (discussed later). Currently, there are no helper functions for those (yet).

### Configuration yamls
The pysacsnow models use yaml files as general run configurations. These yaml files contain information such as start and end dates, time step, and paths to input and output files. The yaml config files are read into pysacsnow using a set of [Pydantic](https://docs.pydantic.dev/latest/) models. To figure out the inputs needed for the yaml configuration, we can use the pydantic classes themselves to help inform us (once again, this isn't necessary, you can also just look at the example yaml files too).

In [24]:
# Print the fields in the config file pydantic class object
print("ConfigFile pydantic model:")
display(ConfigFile.model_fields)
print("Simulation pydantic model:")
display(pysacsnow.base.data_models.Simulation.model_fields)
print("ModelConfig pydantic model:")
display(pysacsnow.base.data_models.ModelConfig.model_fields)
print("Parameter pydantic model:")
display(pysacsnow.base.data_models.Parameter.model_fields)
print("Forcing pydantic model:")
display(pysacsnow.base.data_models.Forcing.model_fields)


ConfigFile pydantic model:


{'simulation': FieldInfo(annotation=Simulation, required=True),
 'model': FieldInfo(annotation=Union[ModelConfig, NoneType], required=False, default=None)}

Simulation pydantic model:


{'name': FieldInfo(annotation=str, required=True),
 'output_path': FieldInfo(annotation=Path, required=True),
 'start_date': FieldInfo(annotation=datetime, required=True),
 'end_date': FieldInfo(annotation=datetime, required=True),
 'time_step': FieldInfo(annotation=int, required=True),
 'time_units': FieldInfo(annotation=str, required=True)}

ModelConfig pydantic model:


{'name': FieldInfo(annotation=str, required=True),
 'dimensions': FieldInfo(annotation=Dim, required=True),
 'parameter_type': FieldInfo(annotation=str, required=True),
 'parameters': FieldInfo(annotation=Parameter, required=True),
 'forcing_type': FieldInfo(annotation=str, required=True),
 'forcing': FieldInfo(annotation=List[Forcing], required=True),
 'initial_conditions': FieldInfo(annotation=InitialConditions, required=True),
 'output_variables': FieldInfo(annotation=Union[List[OutputVariables], NoneType], required=False, default=None)}

Parameter pydantic model:


{'path': FieldInfo(annotation=Path, required=True)}

Forcing pydantic model:


{'name': FieldInfo(annotation=str, required=True),
 'path': FieldInfo(annotation=Path, required=True)}

In [27]:
# Generate a model configuration using dictionary (following the pydantic model above)

location_id = '16103000'

config_inputs = {
    'simulation': {
        'name': location_id,
        'output_path': Path(study_dir, f'data/output/output_{location_id}.csv'),
        'start_date': datetime.datetime(2021, 4, 20, 0, 0),
        'end_date': datetime.datetime(2026, 4, 12, 0, 0),
        'time_step': 3600,
        'time_units': 's'
    },
    'model': {
        'name': 'sac',
        'dimensions': {
            'type': 'nhru',
            'size': 1
        },
        'parameter_type': 'json',
        'parameters': {
            'path': Path(study_dir, f'configurations/sac/sac_params_{location_id}.json')
        },
        'forcing_type': 'csv',
        'forcing': [
            {
                'name': location_id,
                'path': Path(study_dir, f'data/forcing/forcing.sac.{location_id}.csv')
            },
        ],
        'initial_conditions': {
            'restart': False,
            'path': Path('restart.nc')
        },
        'output_variables': None
    }
}

# Create a ConfigFile object using the dictionary, and print the model configuration now as a pydantic object
config = ConfigFile(**config_inputs)
config.model_dump()

{'simulation': {'name': '16103000',
  'output_path': PosixPath('/home/kvanwerkhoven/projects/pysac/hawaii/data/output/output_16103000.csv'),
  'start_date': datetime.datetime(2021, 4, 20, 0, 0),
  'end_date': datetime.datetime(2026, 4, 12, 0, 0),
  'time_step': 3600,
  'time_units': 's'},
 'model': {'name': 'sac',
  'dimensions': {'type': 'nhru', 'size': 1},
  'parameter_type': 'json',
  'parameters': {'path': PosixPath('/home/kvanwerkhoven/projects/pysac/hawaii/configurations/sac/sac_params_16103000.json')},
  'forcing_type': 'csv',
  'forcing': [{'name': '16103000',
    'path': PosixPath('/home/kvanwerkhoven/projects/pysac/hawaii/data/forcing/forcing.sac.16103000.csv')}],
  'initial_conditions': {'restart': False, 'path': PosixPath('restart.nc')},
  'output_variables': None}}

In [28]:
# Use the model_utils module to write the model configuration to a yaml file
pysacsnow.model_utils.create_new_models(
    base_dir=study_dir, 
    config_files=[config]
)

Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/data/forcing
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/data/output
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/sac
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/snow17
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/unithg
Created folder: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/multi-model
ConfigFile written to: /home/kvanwerkhoven/projects/pysac/hawaii/configurations/sac/sac.control.16103000.yaml


In the last cell, we ran the `create_new_models` function again, except this time we passed it a list of `ConfigFile` objects. Using these objects, the function wrote the yaml file for the `sac` model (which was specified in the configuration) to the appropriate folder. This is the first step in setting a new model!